# Native six-bit walk reference (qiskit source)

The original Notebook contains only commented code. This runnable migration uses
FlagQuantum exclusively; it does not execute or depend on the original SDK.

The source shift is modular increment. Coin=1 applies increment; coin=0 applies its inverse. The logical value is six-bit unsigned, with source array index zero as the low bit.

Target basis: CX/U3. U3 is the three-angle U gate. All explicit conversion rules preserve absolute phase.

The source requests U/CX with Qiskit optimization level 3. We replace that private optimization pipeline with a bounded FlagQuantum decomposition; no optimization-level or timing equivalence is claimed.


In [ ]:
from typing import Annotated

import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import (
    H,
    X,
    Output,
    QArray,
    QBit,
    QNum,
    adjoint,
    allocate,
    control,
    qfunc,
)


@qfunc
def increment(x: QNum) -> None:
    x += 1


@qfunc
def decrement(x: QNum) -> None:
    x -= 1


@qfunc
def walk(x: Output[Annotated[QNum, 6]], coin: Output[QBit]) -> None:
    allocate(x)
    allocate(coin)
    H(coin)
    control(coin == 1, lambda: increment(x))
    control(coin == 0, lambda: decrement(x))


program = synth.synthesize(walk, max_qubits=16, max_workspace=9, max_operations=20_000)
result = fq.run(program)
print(dict(program.probabilities(result, registers=("x", "coin"))))


In [ ]:
from examples.synthesis.tutorials.technology_demonstrations.classiq_paper.quantum_walk.reference_workflows import (
    run_workflow,
)

workflow = run_workflow("qiskit")
print(workflow.resources)
print("synthesis_seconds", workflow.synthesis_seconds)
print("lowering_seconds", workflow.lowering_seconds)
print("local_workflow_seconds", workflow.elapsed_seconds)


In [ ]:
from examples.synthesis.tutorials.technology_demonstrations.classiq_paper.quantum_walk.reference_circuit import (
    reference_circuit,
)
from examples.synthesis.tutorials.technology_demonstrations.classiq_paper.quantum_walk.reference_workflows import (
    logical_state,
)
import torch

expected = fq.run(reference_circuit(catalyst=False)).statevector().flatten()
for executable in (program.circuit, workflow.executable):
    torch.testing.assert_close(
        logical_state(executable, program), expected, atol=1e-11, rtol=0
    )
print("independent_full_state_check", "passed")


Source outputs are empty in version control. Tests also check every logical
input column and complex coherent inputs; this initial-state comparison alone
does not prove phase or cleanup. No original compiler timing is reproduced.
